# Layer test: DEM (elevation / slope)

Ingest and display the **SRTM** DEM used in `761_Lab1.ipynb` (`USGS/SRTMGL1_003`).

Low / flat masking is a **project** GIS prior, not a 761 exercise. Elevation and slope cutoffs below are placeholders — change them and record the values in the report.


In [ ]:
import sys
from pathlib import Path

import ee
import geemap

_root = Path.cwd()
_nb = _root / "notebooks" if (_root / "notebooks" / "layer_config.py").exists() else _root
sys.path.insert(0, str(_nb))
import layer_config as cfg

ee.Initialize(project=cfg.GEE_PROJECT)

aoi = ee.Geometry.Rectangle(cfg.AOI_BOUNDS)
print("GEE project:", cfg.GEE_PROJECT)
print("AOI:", cfg.AOI_BOUNDS)

In [ ]:
dem = ee.Image("USGS/SRTMGL1_003").clip(aoi).rename("elevation")
slope = ee.Terrain.slope(dem).rename("slope")

# Project choices (not from a lab). Auckland coastal test defaults:
Z_MAX = 15   # metres
S_MAX = 5    # degrees
low_flat = dem.lt(Z_MAX).And(slope.lt(S_MAX)).rename("low_flat")

print("DEM:", dem.getInfo()["id"])
print("Keep pixels with elevation <", Z_MAX, "m and slope <", S_MAX, "deg")

In [ ]:
dem_vis = {
    "min": 0,
    "max": 300,
    "palette": ["006633", "E5FFCC", "662A00", "D8D8D8", "F5F5F5"],
}
slope_vis = {"min": 0, "max": 20, "palette": ["white", "orange", "red"]}

Map = geemap.Map(center=cfg.MAP_CENTER, zoom=cfg.MAP_ZOOM, basemap="HYBRID")
Map.addLayer(dem, dem_vis, "SRTM elevation")
Map.addLayer(slope, slope_vis, "Slope")
Map.addLayer(low_flat.selfMask(), {"palette": ["#00ffff"]}, "Low / flat mask")
Map.addLayer(aoi, {"color": "red"}, "AOI")
Map